# 🚀 Video Pilot — Google Colab Remote Worker (GPU Tesla T4)

Notebook này biến Google Colab thành **Remote Worker** tự động cho dự án **Video Pilot**.

### Tính năng:
1. **Remotion Video Rendering**: Render video 16:9 1080p bằng GPU NVENC với concurrency cao, giảm 100% gánh nặng CPU/RAM cho máy local.
2. **Expressive Neural TTS**: Chạy mô hình giọng đọc dung lượng lớn với nhịp thở và ngắt nghỉ kịch tính.

---
### Cách chạy:
1. Chọn **Runtime** -> **Change runtime type** -> **T4 GPU**.
2. Bấm **Runtime** -> **Run all** (hoặc `Ctrl + F9`).

In [ ]:
# [1] KIỂM TRA PHẦN CỨNG GPU
!nvidia-smi

In [ ]:
# [2] TẢI MÃ NGUỒN VÀ CÀI ĐẶT MÔI TRƯỜNG
import os, sys

# Clone hoặc cập nhật mã nguồn dự án
!git clone -b feature/colab-offload https://github.com/hongphuoc6104/tiktok-ytb.git /content/tiktok-ytb 2>/dev/null || (cd /content/tiktok-ytb && git pull)
os.chdir('/content/tiktok-ytb/sys')
sys.path.insert(0, '/content/tiktok-ytb/sys')

# Cài đặt Node.js 20.x và thư viện cho Remotion
!apt-get update -qq && apt-get install -y -qq ffmpeg
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
!apt-get install -y nodejs -qq
!npm install --silent

# Cài đặt Cloudflared để tạo Quick Tunnel 0-setup
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true

# Cài đặt các thư viện Python hỗ trợ TTS
!pip install -q soundfile numpy scipy vieneu
print('✅ Đã chuẩn bị xong toàn bộ môi trường và mã nguồn!')

In [ ]:
# [3] KẾT NỐI MẠNG (Tùy chọn: Tailscale hoặc Cloudflare Tunnel)
TAILSCALE_AUTH_KEY = ""  # Điền tskey-auth-... nếu dùng Tailscale

if TAILSCALE_AUTH_KEY:
    !curl -fsSL https://tailscale.com/install.sh | sh
    !tailscale up --authkey=$TAILSCALE_AUTH_KEY --hostname=colab-worker
    !tailscale ip -4
else:
    print("ℹ️ Sử dụng Cloudflare Quick Tunnel (0-setup)")

In [ ]:
# [4] KHỞI ĐỘNG WORKER SERVER & TẠO TUNNEL
import subprocess, time, re

tunnel_url = None
if not TAILSCALE_AUTH_KEY:
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8088'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for _ in range(30):
        line = tunnel_proc.stdout.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
        time.sleep(0.5)

print('=' * 60)
if tunnel_url:
    print('🔗 ĐỊA CHỈ KẾT NỐI COLAB CHO MÁY LOCAL:')
    print(tunnel_url)
    print('=' * 60)
    print('Trên máy local, hãy chạy lệnh:')
    print(f'export COLAB_ENDPOINT="{tunnel_url}"')
else:
    print('🌐 Worker đang lắng nghe trên cổng 8088')
print('=' * 60)

# Chạy Worker Server trên port 8088
!python3 -m colab_bridge.server --port 8088